# CP201A Lab 3: ACS Data in Python

**Fall 2026**

Today your Census API key goes to work. By the end of lab you will have pulled the
same ACS table at three different geographic scales, straight from the Census Bureau
into Python.

## Learning objectives

**Everyone**
* Save your Census API key once, so every notebook this semester can find it
* Read the anatomy of a Census API call: table, variables, geography, year
* Pull an ACS table at the census tract, city, and county scale
* Rename the Census Bureau's variable codes into labels you can actually read
* Check that what came back is what you asked for

**If you want more**
* Pull a whole state's worth of places in one call
* Reshape and export your results for later use

Next week (Lab 4) we take these estimates and deal with their margins of error.

## 0. Before we begin

This notebook uses the `census` package. Run the installation cell first. It will make
sure the package is available in your current DataHub environment. Then run the import
cell below it.

In [ ]:
%pip install -q census

In [ ]:
from census import Census
import pandas as pd
import numpy as np
import os

## 0.1 Saving your Census API key

You only need to do this once all semester. This cell saves your key to a small text
file in your home directory. Every notebook we use from here on will read your key from
that file automatically, so you never have to paste it again.

Before you run this cell, make sure you signed up for a key at
https://api.census.gov/data/key_signup.html and clicked the activation link in the
confirmation email from the Census Bureau.

**To run it:** click the cell, press Shift+Enter, and a prompt will appear below asking
for your key. Paste it in and press Enter. Nothing will appear as you paste, because the
input is hidden on purpose. That is normal.

Pasted the wrong thing? Just run the cell again. It overwrites the old file.

**Why we do it this way:** your key is yours. Keeping it out of the notebook means you
can share, submit, or post a notebook without handing your key to anyone. Later in your
career you will handle keys that really matter, and this is the habit that protects them.

In [ ]:
from getpass import getpass

key = getpass('Paste your Census API key and press Enter: ')

with open(os.path.expanduser('~/census_key.txt'), 'w') as f:
    f.write(key.strip())

print('Key saved. You will not need to do this again.')

Now let's confirm it worked. This next cell is the one that appears at the top of every
later notebook in this course. It reads your key from the file and hands it to the
`Census` object we will use to make requests.

In [ ]:
# This is the same code every later notebook will use to load your key.
with open(os.path.expanduser('~/census_key.txt')) as f:
    api_key = f.read().strip()

print('Key loaded. It starts with:', api_key[:4] + '...')

c = Census(key=api_key)

### Key not working yet?

If your key has not arrived, or you have not clicked the activation link, you can still do
the whole lab. Keep reading, and when you reach Section 2 there is a cell that loads a
saved copy of today's data instead of calling the API.

Get your key sorted before Lab 4, when you will be pulling your own tables.

## 0.2 Two Python tools we will use today

Before the census data, two small things that will show up all afternoon.

### f-strings

An f-string lets you drop the value of a variable into the middle of a piece of text.
Put an `f` before the opening quote, then wrap the variable in curly braces.

In [ ]:
neighborhood = 'North Oakland'
print(f'Today we are looking at {neighborhood}.')

n_tracts = 5
print(f'{neighborhood} is made up of {n_tracts} census tracts.')

In [ ]:
# EXERCISE: try the line above without the f before the quote. What changes?

### for loops

A `for` loop repeats a block of code once for each item in a list. This is how we will
avoid copying and pasting the same line ten times with different variable names.

Read `for item in my_list:` as "for each item in my list, do the following."

In [ ]:
groups = ['White', 'Black', 'Asian', 'Hispanic or Latino']

for g in groups:
    print(f'We will need a column for {g}.')

In [ ]:
# EXERCISE: write a for loop that prints the numbers 1 through 5, each on its own line.
# Hint: range(1, 6) gives you those numbers.

## 1. The anatomy of a Census API call

Every request to the Census API answers four questions:

| Question | In our code | Today's answer |
| --- | --- | --- |
| Which survey? | `c.acs5` | ACS 5-year estimates |
| Which variables? | a list of variable codes | Table B03002 |
| Which geography? | the `for` and `in` arguments | census tracts, in Alameda County, in California |
| Which year? | `year=` | 2024, meaning the 2020 to 2024 5-year estimates |

Two things worth knowing about that last row. The ACS 5-year estimates are labeled by
their **final** year, so `year=2024` gets you data pooled from 2020 through 2024. And
census geographies are **nested**: a tract sits inside a county, which sits inside a
state. That is why the request has both a `for` (the level you want) and an `in` (the
larger thing it sits inside).

### 1.1 Naming the variables

Census variable codes are precise and unreadable. `B03002_003E` is the estimate of
people who are not Hispanic or Latino and identify as White alone. The `E` on the end
means estimate; the same code ending in `M` gives the margin of error.

We are going to build a **dictionary** that maps each code to a label we can read. A
dictionary pairs a key with a value, written `{key: value}`. We will use it twice: once
to tell the API which variables we want, and once to rename the columns that come back.

Below is the standard breakdown of table B03002, Hispanic or Latino Origin by Race.
Everyone of Hispanic or Latino origin is counted as Hispanic, and every race category is
then non-Hispanic. That is a choice, and it is the most common one in planning practice.
You could code it differently. Be deliberate about it, and say what you did.

In [ ]:
variables_of_interest = {
    'NAME': 'NAME',              # no need to rename this one
    'GEO_ID': 'GEO_ID',          # or this one
    'B03002_001E': 'total',
    'B03002_001M': 'total_moe',
    'B03002_003E': 'nh_white',
    'B03002_003M': 'nh_white_moe',
    'B03002_004E': 'nh_black',
    'B03002_004M': 'nh_black_moe',
    'B03002_005E': 'nh_native',
    'B03002_005M': 'nh_native_moe',
    'B03002_006E': 'nh_asian',
    'B03002_006M': 'nh_asian_moe',
    'B03002_007E': 'nh_pi',
    'B03002_007M': 'nh_pi_moe',
    'B03002_008E': 'nh_1other',
    'B03002_008M': 'nh_1other_moe',
    'B03002_009E': 'nh_multi',
    'B03002_009M': 'nh_multi_moe',
    'B03002_012E': 'hispanic',
    'B03002_012M': 'hispanic_moe',
}

print(f'We are asking for {len(variables_of_interest)} columns.')

Notice that we asked for both the estimate (`E`) and the margin of error (`M`) for every
category. Get in the habit of pulling them together. Next week you will need them.

## 2. Pulling data at three scales

The point of this section is to get the same table three times, at three different
geographic levels, so you can see how the request changes and how the answer changes.

### 2.0 No key? Read this first.

If the key cell above printed "Key loaded," ignore this section and go straight to 2.1.

If it did not, run the cell just below. It loads the same three tables from the
`backup_data` folder that came with this notebook. Then **skip the three pull cells in
2.1, 2.2, and 2.3** and pick up at Section 2.4. Everything after that works the same way.

### 2.1 Census tracts (the neighborhood scale)

Census tracts are the smallest geography the ACS publishes for most tables. Neighborhoods
do not come pre-packaged in census data, so planners build them by choosing a set of
tracts.

We are using five tracts in Alameda County. `state:06` is California and `county:001` is
Alameda County.

In [ ]:
# FALLBACK ONLY: run this if your API key is not working yet.
# It loads the same three tables from file. Then skip to Section 2.4.

df_tracts = pd.read_csv('backup_data/lab3_backup_tracts.csv')
df_city   = pd.read_csv('backup_data/lab3_backup_city.csv')
df_county = pd.read_csv('backup_data/lab3_backup_county.csv')

print(f'Loaded {len(df_tracts)} tracts, {len(df_city)} city, {len(df_county)} county.')
print('Now skip down to Section 2.4.')

In [ ]:
NEIGHBORHOOD_NAME = 'North Oakland'
TRACTS = '400500,400600,400700,400800,400900'
ACS_YEAR = 2024          # 2020 to 2024 5-year estimates

df_tracts = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': f'tract:{TRACTS}', 'in': 'state:06 county:001'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

df_tracts

Sanity check, every time. Did you get the number of rows you expected? Do the names look
like the places you asked for? Are the totals plausible for a census tract (usually a few
thousand people)?

In [ ]:
print(f'Rows: {len(df_tracts)}')
print(f'Columns: {len(df_tracts.columns)}')
df_tracts[['NAME', 'total']]

In [ ]:
# EXERCISE: pick a different set of tracts, or a different county, and pull the same
# table. Alameda is county 001; Contra Costa is 013; San Francisco is 075.
# Give your DataFrame a name of your own, not df_tracts.

### 2.2 A city

Cities are "places" in Census vocabulary. Oakland's place code is 53000. Places sit
inside states, so the `in` argument only needs the state.

In [ ]:
df_city = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'place:53000', 'in': 'state:06'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

df_city

### 2.3 A county

Counties sit directly inside states.

In [ ]:
df_county = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'county:001', 'in': 'state:06'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

df_county

In [ ]:
# EXERCISE: pull the same table for the city your neighborhood sits in, and for its
# county. You will use these as comparison geographies in Assignment 1.

### 2.4 Stop and look

Compare the `total` column across your three DataFrames. The tract totals are in the
thousands, the city in the hundreds of thousands, the county over a million.

Now compare the `total_moe` column across the same three. Notice that the margin of
error does not grow proportionally. As a share of the estimate it gets **smaller** as the
geography gets bigger. That single observation is most of what Lab 4 is about.

In [ ]:
print('Tract level:')
print(df_tracts[['NAME', 'total', 'total_moe']].to_string(index=False))
print()
print('City level:')
print(df_city[['NAME', 'total', 'total_moe']].to_string(index=False))
print()
print('County level:')
print(df_county[['NAME', 'total', 'total_moe']].to_string(index=False))

In [ ]:
# EXERCISE: for each of the three geographies above, calculate the margin of error as a
# percentage of the estimate. Which scale gives you the most precise estimate?
# Hint: (moe / estimate) * 100

## 3. Looking at what you got

Three habits worth building now.

In [ ]:
df_tracts.columns     # what are my columns actually called?

In [ ]:
df_tracts.info()      # what type is each column, and are there missing values?

In [ ]:
df_tracts.describe()  # quick numeric summary

`.columns` is the one that will save you the most time. When you get a `KeyError`, the
reflex is: run `df.columns`, find the real name, copy and paste it.

One thing `.info()` will show you: several of these columns came back as **objects**
(text), not numbers. The Census API returns everything as strings. Before you can do
arithmetic, you have to convert.

In [ ]:
# Convert every column except the identifier columns to numbers
id_cols = ['NAME', 'GEO_ID', 'state', 'county', 'tract']

for col in df_tracts.columns:
    if col not in id_cols:
        df_tracts[col] = pd.to_numeric(df_tracts[col])

df_tracts.info()

Read that loop again. For each column name in the DataFrame, if it is not one of the
identifier columns, convert it to a number. This is the same `for` loop pattern from the
warm-up, doing real work.

In [ ]:
# EXERCISE: do the same conversion for your city and county DataFrames.
# Careful: the city and county DataFrames have different identifier columns than the
# tract one. Run .columns on them first and see.

## 4. A first calculation

Raw counts are hard to compare across places of different sizes. Percentages are usually
what you want. Pandas will divide two whole columns at once.

In [ ]:
df_tracts['pct_hispanic'] = df_tracts['hispanic'] / df_tracts['total'] * 100

df_tracts[['NAME', 'total', 'hispanic', 'pct_hispanic']]

In [ ]:
# EXERCISE: calculate the percentage for two more groups in your own DataFrame.

You have just calculated a percentage with no margin of error attached to it. That number
is an estimate built from two other estimates, and it carries uncertainty you cannot see
here. Hold that thought. Next week we put the error bars back on.

## 5. Saving your work

Write your results out so you do not have to re-pull them every time.

In [ ]:
df_tracts.to_csv('lab3_tracts.csv', index=False)
df_city.to_csv('lab3_city.csv', index=False)
df_county.to_csv('lab3_county.csv', index=False)

print('Saved. Check the file browser on the left.')

## 6. If you want more

### 6.1 Every place in California in one call

The `*` wildcard means "all of them."

In [ ]:
df_all_places = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'place:*', 'in': 'state:06'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

print(f'{len(df_all_places)} places in California.')
df_all_places.head()

### 6.2 Finding variables without guessing

You do not have to memorize variable codes. The full list for any year lives at a URL you
can read in your browser:

https://api.census.gov/data/2024/acs/acs5/variables.html

You can also pull the whole variable list into pandas and search it.

In [ ]:
var_url = f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5/groups/B03002.json'
var_table = pd.read_json(var_url)
var_table.head(10)

In [ ]:
# EXERCISE: find a table that interests you for Assignment 1 (median household income is
# B19013, tenure is B25003, means of transportation to work is B08301) and pull it for
# your neighborhood.

### 6.3 A note on the -666666666 values

If you pull enough tables you will eventually see values like -666666666 or -222222222 in
a margin of error column. These are not real numbers. The Census Bureau uses them as
flags: the estimate is controlled, or the sample was too small to compute an MOE, or the
value is not applicable.

Never do arithmetic on them. We will handle them properly in Lab 4.

## Before you leave

* Your key is saved. You will not be asked for it again.
* You can pull an ACS table at three scales.
* You know which table you want for Assignment 1, or you have a short list.

**Coming up:** your Python Basics Task Set from Lab 2 and the Lab 3 check-in are both due
**Sunday, September 13**. Your census tracts and ACS table list for Assignment 1 are due
**Tuesday, September 22**, along with the Measuring Urban Change field trip observation
exercise. Lab 4 (September 23 and 25) takes the estimates you pulled today and works out
what their margins of error mean, so come with your tracts identified.